# CNN Medium Mel-Spectrogram
This notebook downloads and processes only the medium mel-spectrogram dataset.

## Colab Runtime Setup
Local imports work in Colab only if `spectrogram_pipeline.py` is present in the remote runtime. If you use the VS Code Colab extension, make sure the file is synced, uploaded, mounted from Drive, or downloaded before this import runs.

In [ ]:
import sys
from pathlib import Path

PIPELINE_GITHUB_RAW_URL = ""  # Optional: set to raw GitHub URL for spectrogram_pipeline.py
MODULE_FILE = "spectrogram_pipeline.py"

candidate_dirs = [
    Path.cwd(),
    Path("/content"),
    Path("/content/drive/MyDrive/STUDA/src"),
]

module_path = next(
    (directory / MODULE_FILE for directory in candidate_dirs if (directory / MODULE_FILE).exists()),
    None,
)

if module_path is None and PIPELINE_GITHUB_RAW_URL:
    import urllib.request
    module_path = Path("/content") / MODULE_FILE
    urllib.request.urlretrieve(PIPELINE_GITHUB_RAW_URL, module_path)

if module_path is None or not module_path.exists():
    raise FileNotFoundError(
        f"{MODULE_FILE} was not found in the Colab runtime. "
        "Sync it with the VS Code Colab extension, upload it to /content, mount Drive, "
        "or set PIPELINE_GITHUB_RAW_URL before importing."
    )

sys.path.insert(0, str(module_path.parent))
print(f"Using pipeline module: {module_path}")


In [ ]:
from IPython.display import display
from google.colab import files, userdata

from spectrogram_pipeline import (
    build_models_for_variants,
    evaluate_models_for_variants,
    extract_zip,
    plot_training_histories,
    prepare_dataset_variants,
    save_artifacts,
    train_models_for_variants,
    zip_artifacts,
)


## Dataset Download

In [ ]:
DATASET_KEY = "medium"
DATASET_LABEL = "Mel-Spectrogram Medium"
DATASET_SLUG = "pawedyrda/mel-spectrogram-medium"
ARCHIVE_PATH = Path("/content/mel-spectrogram-medium.zip")
DATA_PATH = Path("/content/mel-spectrogram-medium")
EPOCHS = 50
BATCH_SIZE = 32


In [ ]:
kaggle_token = userdata.get("Kaggle")
if kaggle_token is None:
    raise ValueError("Missing Colab secret named 'Kaggle'.")

kaggle_dir = Path("/root/.kaggle")
kaggle_dir.mkdir(parents=True, exist_ok=True)
token_path = kaggle_dir / "access_token"
token_path.write_text(kaggle_token)
token_path.chmod(0o600)


In [ ]:
!kaggle datasets download -d {DATASET_SLUG} -p /content --force
extract_zip(ARCHIVE_PATH, "/content")
print(f"Dataset extracted to: {DATA_PATH}")


## Prepare Normal, M, And W Variants

In [ ]:
variants = prepare_dataset_variants(DATA_PATH)
print("Normal train shape:", variants.normal.train_data.shape)
print("M train shape:", variants.m.train_data.shape)
print("W train shape:", variants.w.train_data.shape)


## Multilabel CNNs

In [ ]:
multilabel_models = build_models_for_variants(variants, model_type="multilabel")
multilabel_histories = train_models_for_variants(
    multilabel_models,
    variants,
    model_type="multilabel",
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
)


## Binary UUV CNNs

In [ ]:
binary_models = build_models_for_variants(variants, model_type="binary")
binary_histories = train_models_for_variants(
    binary_models,
    variants,
    model_type="binary",
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
)


## Evaluation

In [ ]:
multilabel_results = evaluate_models_for_variants(
    multilabel_models, variants, model_type="multilabel", dataset_label=DATASET_LABEL
)
display(multilabel_results)

binary_results = evaluate_models_for_variants(
    binary_models, variants, model_type="binary", dataset_label=DATASET_LABEL
)
display(binary_results)


## Training Curves

In [ ]:
plot_training_histories(multilabel_histories, f"Multilabel {DATASET_LABEL}")
plot_training_histories(binary_histories, f"Binary UUV {DATASET_LABEL}")


## Save Artifacts

In [ ]:
save_dir = save_artifacts(
    save_dir="saved_artifacts",
    dataset_key=DATASET_KEY,
    multilabel_models=multilabel_models,
    binary_models=binary_models,
    multilabel_histories=multilabel_histories,
    binary_histories=binary_histories,
    multilabel_results=multilabel_results,
    binary_results=binary_results,
)
archive_path = zip_artifacts(save_dir, f"saved_models_and_results_{DATASET_KEY}.zip")
files.download(str(archive_path))
